In [ ]:
!pip3 install git+https://github.com/NetManAIOps/sktime.git

In [ ]:
# Time Series Forecasting with sktime Transformer forecaster
# This notebook uses sktime APIs only (no direct torch model code)
# Model: LTSFTransformerForecaster

In [ ]:
# Optional dependency install
# !pip install -U sktime torch

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sktime.datasets import load_airline
from sktime.forecasting.base import ForecastingHorizon
from sktime.forecasting.ltsf import LTSFTransformerForecaster, LTSFLinearForecaster
from sktime.performance_metrics.forecasting import mean_absolute_error, mean_absolute_percentage_error

In [ ]:
print("Loading data...")
airline = load_airline()
print("Dataset shape:", airline.shape)
print(airline.head())
airline.tail()

In [ ]:
plt.figure(figsize=(12, 6))
airline.plot()
plt.title('Airline Passenger Data')
plt.xlabel('Time')
plt.ylabel('Passengers')
plt.grid(True)
plt.show()

In [ ]:
split_point = int(len(airline) * 0.8)
y_train, y_test = airline.iloc[:split_point], airline.iloc[split_point:]

In [ ]:
fh = ForecastingHorizon(y_test.index, is_relative=False)
pred_len = len(y_test)
seq_len = 24
context_len = 12
print(f"Train={len(y_train)}, Test={len(y_test)}, seq_len={seq_len}, pred_len={pred_len}")

In [ ]:
# Main transformer forecaster
transformer_fcst = LTSFTransformerForecaster(
    seq_len=seq_len,
    context_len=context_len,
    pred_len=pred_len,
    num_epochs=20,
    batch_size=8,
    d_model=64,
    n_heads=4,
    d_ff=128,
    e_layers=2,
    d_layers=1,
    dropout=0.1,
    lr=0.001
)

In [ ]:
# Fallback baseline (also from sktime)
linear_fcst = LTSFLinearForecaster(
    seq_len=seq_len,
    pred_len=pred_len,
    num_epochs=20,
    batch_size=8,
    lr=0.001
)

In [ ]:
print("Training LTSFTransformer...")
transformer_fcst.fit(y_train, fh=fh)
y_pred_transformer = transformer_fcst.predict(fh)

print("Training LTSFLinear baseline...")
linear_fcst.fit(y_train, fh=fh)
y_pred_linear = linear_fcst.predict(fh)

In [ ]:
mae_transformer = mean_absolute_error(y_test, y_pred_transformer)
mape_transformer = mean_absolute_percentage_error(y_test, y_pred_transformer)
mae_linear = mean_absolute_error(y_test, y_pred_linear)
mape_linear = mean_absolute_percentage_error(y_test, y_pred_linear)

print(f"Transformer MAE: {mae_transformer:.2f}, MAPE: {mape_transformer:.2%}")
print(f"Linear      MAE: {mae_linear:.2f}, MAPE: {mape_linear:.2%}")

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(y_train.index, y_train.values, label="Train")
plt.plot(y_test.index, y_test.values, label="Test")
plt.plot(y_pred_transformer.index, y_pred_transformer.values, label="LTSFTransformer")
plt.plot(y_pred_linear.index, y_pred_linear.values, label="LTSFLinear Baseline")
plt.title("Transformer Forecasting with sktime")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
future_steps = 12
future_fh = np.arange(1, future_steps + 1)
future_pred_transformer = transformer_fcst.predict(future_fh)
future_pred_linear = linear_fcst.predict(future_fh)

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(airline.index, airline.values, label="Historical")
plt.plot(future_pred_transformer.index, future_pred_transformer.values, "r--", label="Future Transformer")
plt.plot(future_pred_linear.index, future_pred_linear.values, "g--", label="Future Linear")
plt.title("Future Forecast (next 12 steps)")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
print("Summary:")
print(f"LTSFTransformer -> MAE={mae_transformer:.2f}, MAPE={mape_transformer:.2%}")
print(f"LTSFLinear      -> MAE={mae_linear:.2f}, MAPE={mape_linear:.2%}")